In [ ]:
import openai 
import instructor
from pydantic import BaseModel, Field
import pandas as pd

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import VectorParams, Filter, FieldCondition, MatchValue, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery

## Qdrant collection for hybrid search

In [3]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [9]:
collection_name="Amazon-shopping-collection-01-hybrid-search"
qdrant_client.create_collection(
    collection_name = collection_name,
    vectors_config={
        "text-embedding-3-small": VectorParams(size=1536, distance= Distance.COSINE)
    },
    sparse_vectors_config={
        "bm25": SparseVectorParams(modifier=Modifier.IDF)
    }
)

True

In [11]:
qdrant_client.create_payload_index(
    collection_name=collection_name,
    field_name="parent_asin",
    field_schema=PayloadSchemaType.KEYWORD
    )

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

### Embedding Function

In [12]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

### Embeddings for batch

In [14]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model = model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i: i + batch_size]
        response = openai.embeddings.create(input=batch, model = model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    return all_embeddings

In [15]:
df_items = pd.read_json("../../data/meta_Electronics_with_category_sample_1000.jsonl",lines=True)

In [16]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Electronics,"FENIFOX Bluetooth Mouse, Slim Mini Portable Fl...",4.1,2434,[【LIGHT and ULTRA-THIN】: Slim and light design...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Good for travelers but not for reg...,FENIFOX,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '4.52 x 1.97 x 0.55 inc...,B06ZZJ6C3P,NaN,NaN,None
1,Computers,Macally USB Wired Keyboard and Mouse Combo wit...,4.4,1468,"[Low-profile keys give you a quiet, responsive...","[Wired Keyboard and Mouse Combo, The Macally M...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Full Size USB Keyboard and Optical...,Macally,"[Electronics, Computers & Accessories, Compute...","{'Number of USB 2.0 Ports': '2', 'Brand': 'Mac...",B072VHJN54,NaN,NaN,None
2,Computers,AiSMei Case for 9.7-Inch iPad 5th (2017)/ iPad...,4.2,331,"[Specially designed for ipad A1458, A1459, A14...",[Product Description Size: Classical Rotating ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],AiSMei,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '9.7 Inches',...",B013R9SBOC,NaN,NaN,None
3,Camera & Photo,Fujifilm Instax Mini Instant Film Black (10 Sh...,4.7,837,[Express your creativity with this exclusive P...,"[This single pack of, FUJIFILM INSTAX Mini Bla...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Ive been buying this film for 4 ye...,Fujifilm,"[Electronics, Camera & Photo, Film Photography...",{'Package Dimensions': '7.01 x 4.41 x 1.38 inc...,B07WHHYHC8,NaN,NaN,None
4,Computers,KVAGO Keyboard Case for HD 10 and HD 10 Plus T...,4.4,308,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Bluetooth Keyboard for Fire HD 10 ...,KVAGO,"[Electronics, Computers & Accessories, Tablet ...",{'Package Dimensions': '11.02 x 7.91 x 1.18 in...,B09MM4S4XR,NaN,NaN,None


In [17]:
def preprocess_description(row):
    return f"{row['title']}{' '.join(row['features']) }"

In [18]:
def extract_first_large_image(row):
    images = row['images']
    if not images:  # handles None, NaN, and empty list
        return ""
    return images[0].get("large", "")

In [21]:
df_items['preprocessed_description'] = df_items.apply(preprocess_description, axis = 1)
df_items['image'] = df_items.apply(extract_first_large_image, axis = 1)
df_items

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,preprocessed_description,image
0,All Electronics,"FENIFOX Bluetooth Mouse, Slim Mini Portable Fl...",4.1,2434,[【LIGHT and ULTRA-THIN】: Slim and light design...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Good for travelers but not for reg...,FENIFOX,"[Electronics, Computers & Accessories, Compute...",{'Product Dimensions': '4.52 x 1.97 x 0.55 inc...,B06ZZJ6C3P,NaN,NaN,None,"FENIFOX Bluetooth Mouse, Slim Mini Portable Fl...",https://m.media-amazon.com/images/I/31Gg2-xED7...
1,Computers,Macally USB Wired Keyboard and Mouse Combo wit...,4.4,1468,"[Low-profile keys give you a quiet, responsive...","[Wired Keyboard and Mouse Combo, The Macally M...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Full Size USB Keyboard and Optical...,Macally,"[Electronics, Computers & Accessories, Compute...","{'Number of USB 2.0 Ports': '2', 'Brand': 'Mac...",B072VHJN54,NaN,NaN,None,Macally USB Wired Keyboard and Mouse Combo wit...,https://m.media-amazon.com/images/I/41JBmHw+U8...
2,Computers,AiSMei Case for 9.7-Inch iPad 5th (2017)/ iPad...,4.2,331,"[Specially designed for ipad A1458, A1459, A14...",[Product Description Size: Classical Rotating ...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],AiSMei,"[Electronics, Computers & Accessories, Tablet ...","{'Standing screen display size': '9.7 Inches',...",B013R9SBOC,NaN,NaN,None,AiSMei Case for 9.7-Inch iPad 5th (2017)/ iPad...,https://m.media-amazon.com/images/I/51IamUJo78...
3,Camera & Photo,Fujifilm Instax Mini Instant Film Black (10 Sh...,4.7,837,[Express your creativity with this exclusive P...,"[This single pack of, FUJIFILM INSTAX Mini Bla...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Ive been buying this film for 4 ye...,Fujifilm,"[Electronics, Camera & Photo, Film Photography...",{'Package Dimensions': '7.01 x 4.41 x 1.38 inc...,B07WHHYHC8,NaN,NaN,None,Fujifilm Instax Mini Instant Film Black (10 Sh...,https://m.media-amazon.com/images/I/41TFHj6hlM...
4,Computers,KVAGO Keyboard Case for HD 10 and HD 10 Plus T...,4.4,308,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Bluetooth Keyboard for Fire HD 10 ...,KVAGO,"[Electronics, Computers & Accessories, Tablet ...",{'Package Dimensions': '11.02 x 7.91 x 1.18 in...,B09MM4S4XR,NaN,NaN,None,KVAGO Keyboard Case for HD 10 and HD 10 Plus T...,https://m.media-amazon.com/images/I/51+EfwT0MR...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Camera & Photo,Powerextra 2 Pack Replacement Batteries and Ch...,4.6,223,[Includes 2 batteries and 1 charger for the Pa...,[],49.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Budget Friendly Dual Batteries And...,Powerextra,"[Electronics, Camera & Photo, Accessories, Bat...",{'Package Dimensions': '6.18 x 4.41 x 1.89 inc...,B01IMSOJAI,NaN,NaN,None,Powerextra 2 Pack Replacement Batteries and Ch...,https://m.media-amazon.com/images/I/41OT5XDQ-h...
996,All Electronics,"Aux Cord for iPhone, [Apple Mfi Certified] 3-i...",4.0,867,[【Built-in Advanced Certified Chip】 The premiu...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Autynie,"[Electronics, Headphones, Earbuds & Accessorie...","{'Brand': 'Autynie', 'Connector Type': 'Auxili...",B07QXX8713,NaN,NaN,None,"Aux Cord for iPhone, [Apple Mfi Certified] 3-i...",https://m.media-amazon.com/images/I/51yO8ckPni...
997,All Electronics,Cable Raceways - Floor Cord Protector Cable Sh...,4.3,725,[1) Multipurpose Not only a protector on the f...,[Cable Raceways - Floor Cord Protector Cable S...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Tenkful,"[Electronics, Accessories & Supplies, Cord Man...",{'Product Dimensions': '60 x 1.19 x 0.45 inche...,B081YTQZSK,NaN,NaN,None,Cable Raceways - Floor Cord Protector Cable Sh...,https://m.media-amazon.com/images/

In [23]:
list(df_items["preprocessed_description"].items())[0]

(0,
 'FENIFOX Bluetooth Mouse, Slim Mini Portable Flat Travel Wireless Mouse Rechargeable Quiet Ultra-Thin Mice Compatible with Laptop,Tablet,Notebook,PC (Silver and White)【LIGHT and ULTRA-THIN】: Slim and light design, only 0.6 in thick, 0.1 lbs weight. Made of ABS+PC material, it doesn’t occupy USB port when used. The lightweight and portable designing is the first choice for travel. 【LOW DECIBEL BUTTON】: Low decibel left and right mouse buttons, Suitable for libraries, conference rooms, coffee shops and other occasions. 【ENERGE SAVING and ENVIRONMENT-FRIENDLY】: The mouse has its own power switch and enter sleep mode automatically. It is the most safe and stable lithium polymer battery, work over 3 weeks after one fully charge , can be charged more than hundred times. 【WIDE COMPATIBILITY】: Compatible with the devices,that have 3.0 Version Bluetooth or higher.Like PC,laptop,tablet.But Not compatible with iphone or ipad. 【EASY to OPERATE】: There just need three steps to connect devices 

In [24]:
data_to_embed = df_items[["preprocessed_description", "image", "rating_number", "price", "average_rating", "parent_asin"]]
data_to_embed

,preprocessed_description,image,rating_number,price,average_rating,parent_asin
0,"FENIFOX Bluetooth Mouse, Slim Mini Portable Fl...",https://m.media-amazon.com/images/I/31Gg2-xED7...,2434,NaN,4.1,B06ZZJ6C3P
1,Macally USB Wired Keyboard and Mouse Combo wit...,https://m.media-amazon.com/images/I/41JBmHw+U8...,1468,NaN,4.4,B072VHJN54
2,AiSMei Case for 9.7-Inch iPad 5th (2017)/ iPad...,https://m.media-amazon.com/images/I/51IamUJo78...,331,NaN,4.2,B013R9SBOC
3,Fujifilm Instax Mini Instant Film Black (10 Sh...,https://m.media-amazon.com/images/I/41TFHj6hlM...,837,NaN,4.7,B07WHHYHC8
4,KVAGO Keyboard Case for HD 10 and HD 10 Plus T...,https://m.media-amazon.com/images/I/51+EfwT0MR...,308,NaN,4.4,B09MM4S4XR
...,...,...,...,...,...,...
995,Powerextra 2 Pack Replacement Batteries and Ch...,https://m.media-amazon.com/images/I/41OT5XDQ-h...,223,49.99,4.6,B01IMSOJAI
996,"Aux Cord for iPhone, [Apple Mfi Certified] 3-i...",https://m.media-amazon.com/images/I/51yO8ckPni...,867,NaN,4.0,B07QXX8713
997,Cable Raceways - Floor Cord Protector Cable Sh...,https://m.media-amazon.com/images/I/51QcMO5k6P...,725,NaN,4.3,B081YTQZSK
998,LifeProof NUUD iPad Mini 1 Waterproof Case - R...,https://m.media-amazon.com/images/I/51VXxMSNpi...,326,NaN,4.3,B00H2AHACC


In [25]:
data_to_embed_dict  = data_to_embed.to_dict(orient="records")
data_to_embed_dict

[{'preprocessed_description': 'FENIFOX Bluetooth Mouse, Slim Mini Portable Flat Travel Wireless Mouse Rechargeable Quiet Ultra-Thin Mice Compatible with Laptop,Tablet,Notebook,PC (Silver and White)【LIGHT and ULTRA-THIN】: Slim and light design, only 0.6 in thick, 0.1 lbs weight. Made of ABS+PC material, it doesn’t occupy USB port when used. The lightweight and portable designing is the first choice for travel. 【LOW DECIBEL BUTTON】: Low decibel left and right mouse buttons, Suitable for libraries, conference rooms, coffee shops and other occasions. 【ENERGE SAVING and ENVIRONMENT-FRIENDLY】: The mouse has its own power switch and enter sleep mode automatically. It is the most safe and stable lithium polymer battery, work over 3 weeks after one fully charge , can be charged more than hundred times. 【WIDE COMPATIBILITY】: Compatible with the devices,that have 3.0 Version Bluetooth or higher.Like PC,laptop,tablet.But Not compatible with iphone or ipad. 【EASY to OPERATE】: There just need three 

In [26]:
len(data_to_embed_dict)

1000

In [30]:
text_to_embed  = [item["preprocessed_description"] for item in data_to_embed_dict]
text_to_embed

['FENIFOX Bluetooth Mouse, Slim Mini Portable Flat Travel Wireless Mouse Rechargeable Quiet Ultra-Thin Mice Compatible with Laptop,Tablet,Notebook,PC (Silver and White)【LIGHT and ULTRA-THIN】: Slim and light design, only 0.6 in thick, 0.1 lbs weight. Made of ABS+PC material, it doesn’t occupy USB port when used. The lightweight and portable designing is the first choice for travel. 【LOW DECIBEL BUTTON】: Low decibel left and right mouse buttons, Suitable for libraries, conference rooms, coffee shops and other occasions. 【ENERGE SAVING and ENVIRONMENT-FRIENDLY】: The mouse has its own power switch and enter sleep mode automatically. It is the most safe and stable lithium polymer battery, work over 3 weeks after one fully charge , can be charged more than hundred times. 【WIDE COMPATIBILITY】: Compatible with the devices,that have 3.0 Version Bluetooth or higher.Like PC,laptop,tablet.But Not compatible with iphone or ipad. 【EASY to OPERATE】: There just need three steps to connect devices that

In [31]:
embeddings = get_embeddings_batch(text_to_embed)

Processed 100 of 1000
Processed 200 of 1000
Processed 300 of 1000
Processed 400 of 1000
Processed 500 of 1000
Processed 600 of 1000
Processed 700 of 1000
Processed 800 of 1000
Processed 900 of 1000
Processed 1000 of 1000


In [32]:
len(embeddings)

1000

In [33]:
point_structs = []
i  = 1
for embedding, data in zip(embeddings, data_to_embed_dict):
    point_structs.append(
                            PointStruct
                                (id=i,
                                vector={
                                    "text-embedding-3-small": embedding,
                                    "bm25": Document(
                                        text = data["preprocessed_description"],
                                        model="qdrant/bm25"
                                    )
                                    },
                                    payload = data
                                ) 
                        )
    i += 1

In [34]:
point_structs[0].vector

{'text-embedding-3-small': [0.00567626953125,
  0.02691650390625,
  -0.01230621337890625,
  -0.01708984375,
  -0.01873779296875,
  -0.024017333984375,
  -0.035919189453125,
  0.02459716796875,
  0.01070404052734375,
  -0.026275634765625,
  -0.032073974609375,
  0.034881591796875,
  -0.09027099609375,
  0.040283203125,
  0.044952392578125,
  0.0236663818359375,
  -0.0154876708984375,
  -0.01467132568359375,
  -0.001983642578125,
  0.0035533905029296875,
  -0.003879547119140625,
  0.039154052734375,
  0.020660400390625,
  -0.02252197265625,
  0.00495147705078125,
  -0.00768280029296875,
  0.021820068359375,
  -0.034759521484375,
  0.0192108154296875,
  0.0018215179443359375,
  -0.01052093505859375,
  -0.02374267578125,
  0.025787353515625,
  -0.054931640625,
  0.003185272216796875,
  -0.012298583984375,
  0.0271148681640625,
  -0.0538330078125,
  0.01180267333984375,
  -0.035675048828125,
  -0.032196044921875,
  -0.01062774658203125,
  0.042694091796875,
  0.001007080078125,
  0.01681518

In [35]:
qdrant_client.upsert(collection_name = collection_name,
points = point_structs[0:500],
wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [36]:
qdrant_client.upsert(collection_name = collection_name,
points = point_structs[500:],
wait=True
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrieval

In [39]:
def retrieve_context(query, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name, 
        prefetch = [
            Prefetch(query = query_embedding, using = "text-embedding-3-small", limit = 20),
            Prefetch(query = Document(text=query, model = "qdrant/bm25"), using = "bm25", limit = 20)
        ],
        query=FusionQuery(fusion="rrf"), 
        limit=k)

    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

In [48]:
results = retrieve_context("do you have any 4k webcams ?", k = 20)

points=[ScoredPoint(id=826, version=4, score=0.75, payload={'preprocessed_description': 'DEPSTECH Webcam with Microphone, 2K HD Web Camera with Auto Light Correction, Plug and Play USB Webcam and 90° View Angle for Desktop/Streaming/Video Calling Recording/Meeting/Online Teaching【2K QHD Webcam】The HD camera with a 1/2.9”CMOS image sensor delivers sharp and crystal clear video up to a 2560 x 1440 resolution video at a fluid 30 FPS, specially suitable for professional live streaming or recording like detailed makeup tutorials or other online teaching. 【Webcam with Microphone】This web camera built-in microphone with automatic noise reduction make the recording sound clearer, more natural under some noise environments, providing better sound recognition and smooth transmission. 【Smooth Live Streaming Webcam】With automatic low light correction tech, the USB camera captures excellent 2K QHD video at a widescreen angle of up to 90 degrees, even in dimly environment. It also support faster dat

In [49]:
results

{'retrieved_context_ids': ['B08CMV4GF6',
  'B082HQJ1JX',
  'B08R9GPKRH',
  'B087986YPY',
  'B002XJG71M',
  'B0B7QT2KXY',
  'B08PYY2TCC',
  'B09YRZ8VDV',
  'B07SFFT6TQ',
  'B08FBZ6SS3',
  'B0C58RYFHW',
  'B01M4J53CO',
  'B07M9Q2PL4',
  'B078L3Q89G',
  'B08XXQCBSZ',
  'B014RONV5K',
  'B0B5LRMSQX',
  'B0B7BHY16V',
  'B075NFX24N',
  'B0B4Z7V3XG'],
 'retrieved_context': ['DEPSTECH Webcam with Microphone, 2K HD Web Camera with Auto Light Correction, Plug and Play USB Webcam and 90° View Angle for Desktop/Streaming/Video Calling Recording/Meeting/Online Teaching【2K QHD Webcam】The HD camera with a 1/2.9”CMOS image sensor delivers sharp and crystal clear video up to a 2560 x 1440 resolution video at a fluid 30 FPS, specially suitable for professional live streaming or recording like detailed makeup tutorials or other online teaching. 【Webcam with Microphone】This web camera built-in microphone with automatic noise reduction make the recording sound clearer, more natural under some noise environm

### Hybrid Search with weighted RRF

In [43]:
def retrieve_context_weighted(query, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name, 
        prefetch = [
            Prefetch(query = query_embedding, using = "text-embedding-3-small", limit = 20),
            Prefetch(query = Document(text=query, model = "qdrant/bm25"), using = "bm25", limit = 20)
        ],
        query=models.RrfQuery(rrf = models.Rrf(weights=[3,1])),
        limit=k)

    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

In [46]:
results = retrieve_context_weighted("do you have any 4k webcams ?", k = 20)

points=[ScoredPoint(id=826, version=4, score=1.0, payload={'preprocessed_description': 'DEPSTECH Webcam with Microphone, 2K HD Web Camera with Auto Light Correction, Plug and Play USB Webcam and 90° View Angle for Desktop/Streaming/Video Calling Recording/Meeting/Online Teaching【2K QHD Webcam】The HD camera with a 1/2.9”CMOS image sensor delivers sharp and crystal clear video up to a 2560 x 1440 resolution video at a fluid 30 FPS, specially suitable for professional live streaming or recording like detailed makeup tutorials or other online teaching. 【Webcam with Microphone】This web camera built-in microphone with automatic noise reduction make the recording sound clearer, more natural under some noise environments, providing better sound recognition and smooth transmission. 【Smooth Live Streaming Webcam】With automatic low light correction tech, the USB camera captures excellent 2K QHD video at a widescreen angle of up to 90 degrees, even in dimly environment. It also support faster data

In [47]:
results

{'retrieved_context_ids': ['B08CMV4GF6',
  'B082HQJ1JX',
  'B087986YPY',
  'B002XJG71M',
  'B08R9GPKRH',
  'B0B7QT2KXY',
  'B09YRZ8VDV',
  'B07SFFT6TQ',
  'B01M4J53CO',
  'B07M9Q2PL4',
  'B08XXQCBSZ',
  'B0B7BHY16V',
  'B0B4Z7V3XG',
  'B00PC73NQY',
  'B00HGJQUVG',
  'B016F3M7OM',
  'B07QS5NCB3',
  'B08PYY2TCC',
  'B01FFQEP6I',
  'B08N44NZT3'],
 'retrieved_context': ['DEPSTECH Webcam with Microphone, 2K HD Web Camera with Auto Light Correction, Plug and Play USB Webcam and 90° View Angle for Desktop/Streaming/Video Calling Recording/Meeting/Online Teaching【2K QHD Webcam】The HD camera with a 1/2.9”CMOS image sensor delivers sharp and crystal clear video up to a 2560 x 1440 resolution video at a fluid 30 FPS, specially suitable for professional live streaming or recording like detailed makeup tutorials or other online teaching. 【Webcam with Microphone】This web camera built-in microphone with automatic noise reduction make the recording sound clearer, more natural under some noise environm